# Contexta Research — 07 XLMR Prototype Training

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [2]:
from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = Path("/Users/suvratsmacbookair/MyLocalFiles/NLP project")
os.environ["CONTEXTA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SENSEFORGE_DATA_ROOT"] = str(DATA_ROOT)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Data:", DATA_ROOT)

Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Data: /Users/suvratsmacbookair/MyLocalFiles/NLP project


## Real target-token encoder prototype

In [4]:
from senseforge.research import read_jsonl
from collections import Counter

train_rows = read_jsonl(ART / "semcor_train.jsonl")
val_rows = read_jsonl(ART / "semcor_validation.jsonl")
test_rows = read_jsonl(ART / "semcor_test.jsonl")

print("Training:", len(train_rows))
print("Validation:", len(val_rows))
print("Test:", len(test_rows))

print("\nTraining sense distribution:")
for sense, count in sorted(Counter(r["sense_id"] for r in train_rows).items()):
    print(sense, count)

print("\nExample:")
print(train_rows[0])

Training: 146
Validation: 34
Test: 35

Training sense distribution:
bank.1 13
bank.2 10
plant.1 29
plant.2 40
table.1 20
table.2 34

Example:
{'record_id': 'd103.s081.t007', 'text_en': 'To his left , the two skiffs dented their sharp bows into the soft bank .', 'target_word': 'bank', 'target_char_start': 67, 'target_char_end': 71, 'sense_id': 'bank.2', 'source_dataset': 'xl_wsd_semcor_en', 'original_sense_ids': ['bn:00008363n'], 'label_provenance': 'semcor_babelnet_mapping'}


In [5]:
import inspect
from senseforge.neural import WSDDataset, TargetClassifier
from senseforge.training import fit

print("WSDDataset:")
print(inspect.signature(WSDDataset))
print(inspect.getsource(WSDDataset.__getitem__))

print("\nTargetClassifier:")
print(inspect.signature(TargetClassifier))

print("\nTraining function:")
print(inspect.signature(fit))

WSDDataset:
(rows, tokenizer, senses, max_length=160)
 def __getitem__(self,i):
  row=self.rows[i];m=re.search(r'(?<!\w)'+re.escape(row['word'])+r'(?!\w)',row['text'],re.I)
  if m is None:raise ValueError('Target missing from text')
  enc=self.tokenizer(row['text'],truncation=True,max_length=self.max_length,padding='max_length',return_offsets_mapping=True)
  offsets=enc.pop('offset_mapping');mask=[int(a<m.end() and b>m.start()) for a,b in offsets]
  if not any(mask):raise ValueError('Target truncated or tokenizer alignment failed')
  return {**{k:torch.tensor(v) for k,v in enc.items()},'target_mask':torch.tensor(mask,dtype=torch.float32),'labels':torch.tensor(self.senses.index(row['sense_id']))}


TargetClassifier:
(name, num_labels)

Training function:
(model, train_loader, val_loader, out, epochs=8, lr=2e-05, patience=2)


In [6]:
import torch
from torch.utils.data import Dataset

class SemCorWSDDataset(Dataset):
    """SemCor adapter that keeps the annotated target inside a fixed token window."""
    def __init__(self, rows, tokenizer, senses, max_length=96):
        self.rows = rows
        self.tokenizer = tokenizer
        self.sense_to_id = {s: i for i, s in enumerate(senses)}
        self.max_length = max_length

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        text = row["text_en"]
        char_start = int(row["target_char_start"])
        char_end = int(row["target_char_end"])

        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_offsets_mapping=True,
        )
        input_ids = enc["input_ids"]
        offsets = enc["offset_mapping"]

        target_positions = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < char_end and b > char_start
        ]
        if not target_positions:
            raise ValueError(f"Target alignment failed: {row['record_id']}")

        # Keep one start and one end special token; window only over content tokens.
        content_start, content_stop = 1, len(input_ids) - 1
        capacity = self.max_length - 2
        first, last = target_positions[0], target_positions[-1]
        if last - first + 1 > capacity:
            raise ValueError(f"Target too long: {row['record_id']}")

        window_start = max(content_start, first - capacity // 2)
        window_start = min(window_start, max(content_start, last - capacity + 1))
        window_stop = min(content_stop, window_start + capacity)
        if window_stop - window_start < capacity:
            window_start = max(content_start, window_stop - capacity)

        selected = list(range(window_start, window_stop))
        ids = [input_ids[0]] + [input_ids[i] for i in selected] + [input_ids[-1]]
        target_mask = [0.0] + [float(i in target_positions) for i in selected] + [0.0]
        attention_mask = [1] * len(ids)

        pad = self.max_length - len(ids)
        if pad < 0:
            raise RuntimeError(f"Window construction exceeded max_length: {row['record_id']}")
        ids += [self.tokenizer.pad_token_id] * pad
        attention_mask += [0] * pad
        target_mask += [0.0] * pad

        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "target_mask": torch.tensor(target_mask, dtype=torch.float32),
            "labels": torch.tensor(self.sense_to_id[row["sense_id"]], dtype=torch.long),
        }

print("SemCor target-centered dataset adapter ready")


SemCor target-centered dataset adapter ready


### Target-centered truncation
The dataset adapter above tokenizes without truncation, aligns the annotated character span to subword tokens, then builds a fixed-length window around that target. This avoids silently truncating the supervised token and keeps `input_ids`, `attention_mask`, and `target_mask` exactly aligned.


In [8]:
from transformers import AutoTokenizer
from collections import Counter

tokenizer = AutoTokenizer.from_pretrained(
    "xlm-roberta-base",
    use_fast=True
)

senses = sorted({r["sense_id"] for r in train_rows})

for split_name, split_rows in [
    ("train", train_rows),
    ("validation", val_rows),
    ("test", test_rows)
]:
    dataset = SemCorWSDDataset(
        split_rows, tokenizer, senses, max_length=96
    )

    failures = []
    for i in range(len(dataset)):
        try:
            dataset[i]
        except Exception as e:
            failures.append((i, str(e)))

    print(f"{split_name}: {len(split_rows)} records, {len(failures)} failures")
    if failures:
        print("Examples:", failures[:5])

train: 146 records, 0 failures
validation: 34 records, 0 failures
test: 35 records, 0 failures


In [9]:
for split_name, split_rows in [
    ("train", train_rows),
    ("validation", val_rows),
    ("test", test_rows),
]:
    dataset = SemCorWSDDataset(split_rows, tokenizer, senses, max_length=96)
    failures = []

    for i in range(len(dataset)):
        try:
            item = dataset[i]
            assert item["input_ids"].shape == (96,)
            assert item["attention_mask"].shape == (96,)
            assert item["target_mask"].shape == (96,)
            assert item["target_mask"].sum().item() > 0
            assert torch.all(item["target_mask"] <= item["attention_mask"].float())
        except Exception as e:
            failures.append((i, repr(e)))

    print(f"{split_name}: {len(split_rows)} records, {len(failures)} failures")
    if failures:
        print("Examples:", failures[:5])


train: 146 records, 0 failures
validation: 34 records, 0 failures
test: 35 records, 0 failures


In [10]:
import inspect

print(inspect.getsource(SemCorWSDDataset.__getitem__))

    def __getitem__(self, index):
        row = self.rows[index]
        text = row["text_en"]
        char_start = int(row["target_char_start"])
        char_end = int(row["target_char_end"])

        enc = self.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_offsets_mapping=True,
        )
        input_ids = enc["input_ids"]
        offsets = enc["offset_mapping"]

        target_positions = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < char_end and b > char_start
        ]
        if not target_positions:
            raise ValueError(f"Target alignment failed: {row['record_id']}")

        # Keep one start and one end special token; window only over content tokens.
        content_start, content_stop = 1, len(input_ids) - 1
        capacity = self.max_length - 2
        first, last = target_positions[0], target_positions[-1]
        if last - first + 1 > capacity:
            raise

In [11]:
import inspect
from senseforge.neural import TargetClassifier

print(inspect.getsource(TargetClassifier))

class TargetClassifier(nn.Module):
 def __init__(self,name,num_labels):
  super().__init__();from transformers import AutoModel
  self.encoder=AutoModel.from_pretrained(name);self.head=nn.Linear(self.encoder.config.hidden_size,num_labels)
 def forward(self,input_ids,attention_mask,target_mask,labels=None,**kwargs):
  x=self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
  z=(x*target_mask.unsqueeze(-1)).sum(1)/target_mask.sum(1).clamp(min=1).unsqueeze(-1)
  logits=self.head(z);return (nn.functional.cross_entropy(logits,labels),logits) if labels is not None else logits



## Train prototype only with verified labels

In [13]:
import json
import gc
import torch
from torch.utils.data import DataLoader
from senseforge.neural import TargetClassifier
from senseforge.training import fit, training_plot, seed_all

seed_all()

model_name = "xlm-roberta-base"
output_path = ART / "xlmr_prototype_frozen.pt"

# Use existing validated SemCor splits
senses = sorted({r["sense_id"] for r in train_rows})

train_ds = SemCorWSDDataset(
    train_rows, tokenizer, senses, max_length=96
)
val_ds = SemCorWSDDataset(
    val_rows, tokenizer, senses, max_length=96
)

train_loader = DataLoader(
    train_ds, batch_size=1, shuffle=True, num_workers=0
)
val_loader = DataLoader(
    val_ds, batch_size=1, shuffle=False, num_workers=0
)

# Load XLM-R with a six-class classification head
model = TargetClassifier(model_name, len(senses))

# Freeze the entire pretrained encoder
for param in model.encoder.parameters():
    param.requires_grad = False

# Train only the classification head
for param in model.head.parameters():
    param.requires_grad = True

# Keep encoder in evaluation mode

# Keep the frozen encoder in evaluation mode even when fit() calls model.train()
original_train = model.train

def train_with_frozen_encoder(mode=True):
    original_train(mode)
    model.encoder.eval()
    return model

model.train = train_with_frozen_encoder
model.encoder.eval()

trainable = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model:", model_name)
print("Device available:", "mps" if torch.backends.mps.is_available() else "cpu")
print("Trainable parameters:", trainable)
print("Training records:", len(train_ds))
print("Validation records:", len(val_ds))
print("Batch size:", 1)
print("Encoder:", "Frozen")

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

history = fit(
    model,
    train_loader,
    val_loader,
    output_path,
    epochs=2,
    patience=2,
    lr=2e-5
)

(ART / "xlmr_prototype_frozen.labels.json").write_text(
    json.dumps(senses, indent=2),
    encoding="utf-8"
)

training_plot(
    history,
    ART / "xlmr_prototype_frozen.png"
)

print("Training completed")
print("History:", history)
print("Checkpoint:", output_path)

Model: xlm-roberta-base
Device available: mps
Trainable parameters: 4614
Training records: 146
Validation records: 34
Batch size: 1
Encoder: Frozen
Training completed
History: [{'epoch': 1, 'train_loss': 1.8201574258608362, 'val_loss': 1.6532057383481193, 'accuracy': 0.4117647058823529, 'macro_f1': 0.09929078014184396}, {'epoch': 2, 'train_loss': 1.7729099450046069, 'val_loss': 1.6127905950826757, 'accuracy': 0.5294117647058824, 'macro_f1': 0.19999999999999998}]
Checkpoint: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/xlmr_prototype_frozen.pt


In [24]:
import gc
import json
import numpy as np
import torch

from pathlib import Path
from collections import defaultdict
from transformers import AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

MODEL_NAME = "xlm-roberta-base"
DEVICE = torch.device("cpu")
MAX_LENGTH = 96

prototype_dir = ART / "xlmr_sense_prototypes"
prototype_dir.mkdir(parents=True, exist_ok=True)

# Release previous training model if present
if "model" in globals():
    del model

gc.collect()

if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# --------------------------------------------------
# 2. Load frozen XLM-R encoder
# --------------------------------------------------

encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
)

encoder.to(DEVICE)
encoder.eval()

for parameter in encoder.parameters():
    parameter.requires_grad = False

print("Encoder loaded:", MODEL_NAME)
print("Device:", DEVICE)

# --------------------------------------------------
# 3. Extract target-token embeddings
# --------------------------------------------------

@torch.inference_mode()
def extract_embeddings(rows):
    dataset = SemCorWSDDataset(
        rows,
        tokenizer,
        senses,
        max_length=MAX_LENGTH
    )

    embeddings = []
    labels = []

    for i in range(len(dataset)):
        item = dataset[i]

        input_ids = item["input_ids"].unsqueeze(0).to(DEVICE)
        attention_mask = item["attention_mask"].unsqueeze(0).to(DEVICE)
        target_mask = item["target_mask"].unsqueeze(0).to(DEVICE)

        output = encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden = output.last_hidden_state

        target_embedding = (
            hidden * target_mask.unsqueeze(-1)
        ).sum(dim=1) / target_mask.sum(dim=1).clamp(min=1).unsqueeze(-1)

        embeddings.append(
            target_embedding.squeeze(0).cpu().numpy()
        )

        labels.append(rows[i]["sense_id"])

        if (i + 1) % 25 == 0:
            print(f"Processed {i + 1}/{len(dataset)}")

    return np.asarray(embeddings), np.asarray(labels)

# --------------------------------------------------
# 4. Training embeddings
# --------------------------------------------------

print("\nExtracting training embeddings...")

X_train, y_train = extract_embeddings(train_rows)

print("Training embedding shape:", X_train.shape)

# --------------------------------------------------
# 5. Construct sense centroids
# --------------------------------------------------

centroids = {}

for sense in sorted(set(y_train)):
    vectors = X_train[y_train == sense]
    centroid = vectors.mean(axis=0)

    norm = np.linalg.norm(centroid)
    centroids[sense] = centroid / max(norm, 1e-12)

print("\nSense prototypes constructed:", len(centroids))

# --------------------------------------------------
# 6. Validation
# --------------------------------------------------

print("\nExtracting validation embeddings...")

X_val, y_val = extract_embeddings(val_rows)

sense_names = sorted(centroids)
prototype_matrix = np.stack(
    [centroids[s] for s in sense_names]
)

def predict_prototypes(X):
    similarities = cosine_similarity(
        X, prototype_matrix
    )
    indices = similarities.argmax(axis=1)
    return np.asarray([sense_names[i] for i in indices])

val_predictions = predict_prototypes(X_val)

accuracy = accuracy_score(y_val, val_predictions)
macro_f1 = f1_score(
    y_val, val_predictions,
    labels=sense_names,
    average="macro",
    zero_division=0
)

print("\nValidation accuracy:", accuracy)
print("Validation Macro-F1:", macro_f1)

print("\nClassification report:")
print(classification_report(
    y_val,
    val_predictions,
    labels=sense_names,
    zero_division=0
))

# --------------------------------------------------
# 7. Save prototype artifacts
# --------------------------------------------------

np.savez_compressed(
    prototype_dir / "sense_prototypes.npz",
    senses=np.asarray(sense_names),
    centroids=prototype_matrix
)

(prototype_dir / "metadata.json").write_text(
    json.dumps({
        "model": MODEL_NAME,
        "method": "frozen_xlmr_target_embedding_centroids",
        "training_records": len(train_rows),
        "validation_records": len(val_rows),
        "max_length": MAX_LENGTH,
        "validation_accuracy": float(accuracy),
        "validation_macro_f1": float(macro_f1)
    }, indent=2),
    encoding="utf-8"
)

print("\nPrototype artifacts saved:", prototype_dir)

Encoder loaded: xlm-roberta-base
Device: cpu

Extracting training embeddings...
Processed 25/146
Processed 50/146
Processed 75/146
Processed 100/146
Processed 125/146
Training embedding shape: (146, 768)

Sense prototypes constructed: 6

Extracting validation embeddings...
Processed 25/34

Validation accuracy: 0.7647058823529411
Validation Macro-F1: 0.7354497354497355

Classification report:
              precision    recall  f1-score   support

      bank.1       1.00      0.75      0.86         4
      bank.2       0.50      1.00      0.67         1
     plant.1       0.20      0.25      0.22         4
     plant.2       0.81      0.93      0.87        14
     table.1       1.00      1.00      1.00         2
     table.2       1.00      0.67      0.80         9

    accuracy                           0.76        34
   macro avg       0.75      0.77      0.74        34
weighted avg       0.81      0.76      0.77        34


Prototype artifacts saved: /Users/suvratsmacbookair/MyLocalFi

In [26]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Existing embeddings and prototypes are reused.
# No additional XLM-R inference is required.

def predict_lemma_restricted(X, rows):
    predictions = []

    for embedding, row in zip(X, rows):
        word = row["target_word"]

        allowed = [
            sense for sense in sense_names
            if sense.startswith(word + ".")
        ]

        if not allowed:
            raise ValueError(f"No prototypes for word: {word}")

        allowed_vectors = np.stack([
            centroids[sense] for sense in allowed
        ])

        similarities = cosine_similarity(
            embedding.reshape(1, -1),
            allowed_vectors
        )[0]

        predictions.append(
            allowed[int(np.argmax(similarities))]
        )

    return np.asarray(predictions)


restricted_predictions = predict_lemma_restricted(
    X_val, val_rows
)

print("Lemma-restricted validation accuracy:",
      accuracy_score(y_val, restricted_predictions))

print("Lemma-restricted validation Macro-F1:",
      f1_score(
          y_val,
          restricted_predictions,
          labels=sense_names,
          average="macro",
          zero_division=0
      ))

print("\nClassification report:")
print(classification_report(
    y_val,
    restricted_predictions,
    labels=sense_names,
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_val,
    restricted_predictions,
    labels=sense_names
))

Lemma-restricted validation accuracy: 0.8235294117647058
Lemma-restricted validation Macro-F1: 0.7441643323996265

Classification report:
              precision    recall  f1-score   support

      bank.1       1.00      0.75      0.86         4
      bank.2       0.50      1.00      0.67         1
     plant.1       0.50      0.25      0.33         4
     plant.2       0.81      0.93      0.87        14
     table.1       0.67      1.00      0.80         2
     table.2       1.00      0.89      0.94         9

    accuracy                           0.82        34
   macro avg       0.75      0.80      0.74        34
weighted avg       0.83      0.82      0.81        34


Confusion matrix:
[[ 3  1  0  0  0  0]
 [ 0  1  0  0  0  0]
 [ 0  0  1  3  0  0]
 [ 0  0  1 13  0  0]
 [ 0  0  0  0  2  0]
 [ 0  0  0  0  1  8]]


In [28]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import json

# Final held-out evaluation
# No prototype reconstruction or parameter tuning using test data.

print("Extracting held-out test embeddings...")

X_test, y_test = extract_embeddings(test_rows)

test_predictions = predict_lemma_restricted(
    X_test, test_rows
)

test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    labels=sense_names,
    average="macro",
    zero_division=0
)

print("\nHeld-out test accuracy:", test_accuracy)
print("Held-out test Macro-F1:", test_macro_f1)

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=sense_names,
    zero_division=0
))

print("\nConfusion matrix:")
print(confusion_matrix(
    y_test,
    test_predictions,
    labels=sense_names
))

# Save test metrics separately
metrics = {
    "model": "xlm-roberta-base",
    "method": "lemma_restricted_sense_prototypes",
    "train_records": len(train_rows),
    "validation_records": len(val_rows),
    "test_records": len(test_rows),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1)
}

metrics_path = prototype_dir / "heldout_test_metrics.json"

metrics_path.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

print("\nTest metrics saved:", metrics_path)

Extracting held-out test embeddings...
Processed 25/35

Held-out test accuracy: 0.8571428571428571
Held-out test Macro-F1: 0.7964065255731922

Classification report:
              precision    recall  f1-score   support

      bank.1       0.67      0.67      0.67         3
      bank.2       0.67      0.67      0.67         3
     plant.1       0.75      1.00      0.86         3
     plant.2       1.00      0.93      0.96        14
     table.1       0.60      1.00      0.75         3
     table.2       1.00      0.78      0.88         9

    accuracy                           0.86        35
   macro avg       0.78      0.84      0.80        35
weighted avg       0.89      0.86      0.86        35


Confusion matrix:
[[ 2  1  0  0  0  0]
 [ 1  2  0  0  0  0]
 [ 0  0  3  0  0  0]
 [ 0  0  1 13  0  0]
 [ 0  0  0  0  3  0]
 [ 0  0  0  0  2  7]]

Test metrics saved: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge/artifacts/xlmr_sense_prototypes/heldout_test_metrics.json
